# F1 Prediction Model — Exploration & Training

Inspired by the tennis prediction project that achieved 85% accuracy on the Australian Open.
We apply the same approach to Formula 1: ELO ratings + XGBoost.

## Pipeline
1. Ingest 75 years of F1 data (Jolpica API)
2. Build multi-dimensional ELO system
3. Engineer features (form, circuit performance, weather, etc.)
4. Train XGBoost models
5. Evaluate on held-out seasons
6. Predict upcoming races

In [ ]:
import sys
sys.path.insert(0, '..')
sys.path.insert(0, '../..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('dark_background')
sns.set_palette('husl')
%matplotlib inline

## Step 1: Data Ingestion
Download all historical F1 data. This takes ~10-15 minutes the first time (cached after).

In [ ]:
from data.ingest.jolpica import JolpicaClient

client = JolpicaClient(cache=True)

# Start with a smaller range for quick testing, then expand
datasets = client.ingest_all_seasons(start_year=2000, end_year=2025)

for name, df in datasets.items():
    print(f'{name}: {len(df):,} rows × {len(df.columns)} columns')

In [ ]:
results = datasets['race_results']
print(f'Total race entries: {len(results):,}')
print(f'Seasons: {results["season"].min()} - {results["season"].max()}')
print(f'Unique drivers: {results["driver_id"].nunique()}')
print(f'Unique circuits: {results["circuit_id"].nunique()}')
results.head(10)

## Step 2: Build ELO Ratings
Multi-dimensional ELO system adapted from chess.

In [ ]:
from data.features.elo import build_elo_from_history

qualifying = datasets.get('qualifying', pd.DataFrame())

elo = build_elo_from_history(
    race_results=results,
    qualifying=qualifying,
)

print(f'Races processed: {len(elo.races_processed)}')
print(f'Drivers rated: {len(elo.overall)}')
print(f'Constructors rated: {len(elo.constructors)}')

In [ ]:
# Current driver rankings
driver_ratings = elo.get_driver_ratings()
print('Top 20 Drivers by ELO:')
driver_ratings.head(20)

In [ ]:
# Visualize ELO history — top drivers
top_drivers = ['verstappen', 'hamilton', 'leclerc', 'norris', 'alonso', 'sainz', 'piastri', 'russell']

fig, ax = plt.subplots(figsize=(16, 8))

for driver_id in top_drivers:
    history = elo.get_driver_history(driver_id)
    if history:
        ax.plot(history, label=driver_id, linewidth=2)

ax.set_xlabel('Race Number')
ax.set_ylabel('ELO Rating')
ax.set_title('F1 Driver ELO Ratings Over Time')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Constructor rankings
constructor_ratings = elo.get_constructor_ratings()
print('Constructor Rankings:')
constructor_ratings.head(15)

In [ ]:
# Head-to-head predictions
matchups = [
    ('verstappen', 'hamilton'),
    ('verstappen', 'norris'),
    ('leclerc', 'sainz'),
    ('hamilton', 'russell'),
    ('norris', 'piastri'),
]

print('Head-to-Head Predictions (ELO-based):')
print('=' * 60)
for d1, d2 in matchups:
    pred = elo.get_matchup_prediction(d1, d2)
    print(f"{d1} vs {d2}: {pred['prob_a_wins']:.1%} vs {pred['prob_b_wins']:.1%} (ELO diff: {pred['elo_diff']:+.0f})")

## Step 3: Feature Engineering

In [ ]:
from data.features.engineer import build_feature_matrix, prepare_training_data

feature_matrix = build_feature_matrix(
    race_results=results,
    qualifying=qualifying,
    drivers=datasets['drivers'],
    constructors=datasets['constructors'],
    circuits=datasets['circuits'],
)

print(f'Feature matrix: {len(feature_matrix):,} rows × {len(feature_matrix.columns)} columns')
feature_matrix.head()

In [ ]:
# Feature correlation with finishing position
numeric_cols = feature_matrix.select_dtypes(include=[np.number]).columns
correlations = feature_matrix[numeric_cols].corr()['position'].sort_values()

fig, ax = plt.subplots(figsize=(10, 12))
correlations.drop('position').plot(kind='barh', ax=ax)
ax.set_title('Feature Correlation with Finishing Position')
ax.set_xlabel('Correlation')
plt.tight_layout()
plt.show()

## Step 4: Train XGBoost Models

In [ ]:
from data.models.predictor import train_and_evaluate

# Hold out 2024-2025 for testing
model, metrics = train_and_evaluate(
    feature_matrix,
    test_seasons=[2024, 2025],
)

print('\nEvaluation Metrics:')
for k, v in metrics.items():
    print(f'  {k}: {v:.4f}')

In [ ]:
# Feature importance
fig, ax = plt.subplots(figsize=(10, 8))
model.feature_importance.head(20).plot(
    kind='barh', x='feature', y='importance', ax=ax, legend=False
)
ax.set_title('Top 20 Most Important Features')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

In [ ]:
# Save the trained model
model.save()
print('Model saved successfully.')

## Step 5: Predict Next Race
Predict the upcoming race using current driver/constructor ratings and recent form.

In [ ]:
# Get latest features for current grid
latest_season = feature_matrix['season'].max()
latest_round = feature_matrix[feature_matrix['season'] == latest_season]['round'].max()

# Use the latest round's features as a proxy for the next race
latest_features = feature_matrix[
    (feature_matrix['season'] == latest_season) &
    (feature_matrix['round'] == latest_round)
].copy()

if not latest_features.empty:
    predictions = model.predict_race(latest_features)
    print(f'Predictions based on Season {latest_season}, Round {latest_round} features:')
    print('=' * 70)
    display_cols = ['driver_id', 'predicted_rank', 'predicted_position', 'prob_winner', 'prob_podium', 'prob_points']
    available_cols = [c for c in display_cols if c in predictions.columns]
    print(predictions[available_cols].to_string(index=False))
else:
    print('No data available for latest season. Run ingestion first.')